# Task 2.2 EDA — 스마트폰 리뷰 데이터 탐색

목적: 방법 선택 전에 데이터가 무엇을 갖고 있는지 파악 → **질문/가설 도출**

데이터: `si_dataset/review_for_analysis.json`  
- 총 2,757개 리뷰 / 7개 스마트폰 모델 (삼성 4종, 애플 3종)
- 필드: rating, helpfulCount, reviewSurveyAnswers, content, title, reviewAt, itemName, ...

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter
from pathlib import Path

# sns.set_theme 먼저 — 이후에 설정한 폰트를 덮어쓰지 않도록
sns.set_theme(style='whitegrid', palette='muted')

# 한글 폰트 — set_theme 이후에 지정해야 Arial로 리셋되지 않음
_font_path = r'C:\Windows\Fonts\malgun.ttf'
fm.fontManager.addfont(_font_path)
_font_name = fm.FontProperties(fname=_font_path).get_name()
plt.rcParams['font.family'] = _font_name
plt.rcParams['axes.unicode_minus'] = False
# seaborn rc도 같이 덮어쓰기
sns.set_theme(style='whitegrid', palette='muted', rc={
    'font.family': _font_name,
    'axes.unicode_minus': False,
})

print(f'사용 폰트: {plt.rcParams["font.family"]}')

DATA_PATH = Path('si_dataset/review_for_analysis.json')

In [ ]:
# JSON에 NaN, nullDeliberate 등 비표준 값이 포함되어 있어 전처리 후 로드
raw = DATA_PATH.read_text(encoding='utf-8')
raw = re.sub(r'\bNaN\b', 'null', raw)
raw = re.sub(r'\bnullDeliberate\b', 'null', raw)
records = json.loads(raw)

df = pd.json_normalize(records)
print(f'총 레코드: {len(df):,}개')
print(f'컬럼 수: {df.shape[1]}개')
df.head(2)

---
## 1. 기본 구조 확인

In [ ]:
# 분석에 쓸 핵심 컬럼만 정리
df['reviewAt_dt'] = pd.to_datetime(df['reviewAt'], unit='ms')
df['content_len'] = df['content'].fillna('').str.len()
df['brand'] = df['product_name'].apply(
    lambda x: 'Apple' if 'iphone' in str(x) else 'Samsung'
)

# 제품명 짧게
product_labels = {
    'iphone_17': 'iPhone 17',
    'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26',
    'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7',
    'galaxy_z_flip7': 'Galaxy Z Flip7',
}
df['product_label'] = df['product_name'].map(product_labels)

print(df[['product_name', 'brand', 'rating', 'helpfulCount', 'content_len', 'reviewAt_dt']].dtypes)
df[['product_label', 'brand', 'rating', 'helpfulCount', 'helpfulTrueCount', 'helpfulFalseCount', 'content_len']].describe()

In [ ]:
# 결측치 현황
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('결측 있는 컬럼:')
print(missing.to_string())

---
## 2. 제품별 리뷰 수 & 평점 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2-1. 제품별 리뷰 수
product_order = df['product_label'].value_counts().index
colors = ['#4A90D9' if 'iPhone' in p else '#E85D5D' for p in product_order]
ax = axes[0]
df['product_label'].value_counts().reindex(product_order).plot(
    kind='bar', ax=ax, color=colors, edgecolor='white'
)
ax.set_title('제품별 리뷰 수', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_xticklabels(product_order, rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

# 2-2. 전체 평점 분포
ax2 = axes[1]
rating_counts = df['rating'].value_counts().sort_index()
rating_pct = rating_counts / len(df) * 100
bars = ax2.bar(rating_counts.index, rating_counts.values,
               color=['#d9534f','#e88a3d','#f0c040','#5cb85c','#337ab7'],
               edgecolor='white')
ax2.set_title('전체 평점 분포', fontsize=13, fontweight='bold')
ax2.set_xlabel('별점')
ax2.set_xticks([1,2,3,4,5])
for bar, pct in zip(bars, rating_pct):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('output/eda_01_product_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n5점 비율: {rating_pct[5]:.1f}%  — 극단적인 쏠림 주의')

In [ ]:
# 제품별 평점 분포 히트맵
rating_by_product = (
    df.groupby(['product_label', 'rating'])
    .size()
    .unstack(fill_value=0)
)
# 비율로 변환
rating_by_product_pct = rating_by_product.div(rating_by_product.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    rating_by_product_pct,
    annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=0.5, ax=ax, cbar_kws={'label': '%'}
)
ax.set_title('제품별 평점 분포 (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('별점')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('output/eda_02_rating_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. 브랜드별 평점 비교 (Apple vs Samsung)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, brand in zip(axes, ['Apple', 'Samsung']):
    sub = df[df['brand'] == brand]['rating'].value_counts().sort_index()
    pct = sub / sub.sum() * 100
    color = '#4A90D9' if brand == 'Apple' else '#E85D5D'
    bars = ax.bar(sub.index, sub.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'{brand} — 평점 분포', fontsize=12, fontweight='bold')
    ax.set_xticks([1,2,3,4,5])
    ax.set_xlabel('별점')
    for bar, p in zip(bars, pct):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{p:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('브랜드별 평점 비교', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/eda_03_brand_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('brand')['rating'].agg(['mean','median','std']).round(2))

---
## 4. 시계열: 리뷰 시점 분포

In [ ]:
df['review_month'] = df['reviewAt_dt'].dt.to_period('M')

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# 4-1. 월별 전체 리뷰 수
monthly = df.groupby('review_month').size()
ax1 = axes[0]
monthly.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('월별 리뷰 수', fontsize=12, fontweight='bold')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=45)

# 4-2. 월별 브랜드별 리뷰 수
ax2 = axes[1]
monthly_brand = df.groupby(['review_month', 'brand']).size().unstack(fill_value=0)
monthly_brand.plot(kind='bar', ax=ax2, color=['#4A90D9', '#E85D5D'],
                   edgecolor='white', width=0.7)
ax2.set_title('월별 브랜드별 리뷰 수', fontsize=12, fontweight='bold')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='브랜드')

plt.tight_layout()
plt.savefig('output/eda_04_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

print('리뷰 기간:', df['reviewAt_dt'].min().date(), '~', df['reviewAt_dt'].max().date())

---
## 5. Helpful Votes 분석

In [ ]:
df_help = df[df['helpfulCount'] > 0].copy()
print(f'helpfulCount > 0 리뷰: {len(df_help)}개 ({len(df_help)/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 5-1. helpfulCount 분포 (log scale)
axes[0].hist(df_help['helpfulCount'], bins=30, color='teal', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_title('Helpful Count 분포 (log scale)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('도움돼요 수')

# 5-2. 평점별 평균 helpfulCount
avg_help_rating = df.groupby('rating')['helpfulCount'].mean()
axes[1].bar(avg_help_rating.index, avg_help_rating.values,
            color=['#d9534f','#e88a3d','#f0c040','#5cb85c','#337ab7'],
            edgecolor='white')
axes[1].set_title('평점별 평균 도움돼요', fontsize=11, fontweight='bold')
axes[1].set_xlabel('별점')
axes[1].set_xticks([1,2,3,4,5])

# 5-3. 리뷰 길이 vs helpfulCount (샘플)
axes[2].scatter(df_help['content_len'], df_help['helpfulCount'],
                alpha=0.3, s=15, color='purple')
axes[2].set_title('리뷰 길이 vs 도움돼요', fontsize=11, fontweight='bold')
axes[2].set_xlabel('리뷰 글자 수')
axes[2].set_ylabel('helpfulCount')

plt.tight_layout()
plt.savefig('output/eda_05_helpful.png', dpi=150, bbox_inches='tight')
plt.show()

# 가장 많은 도움돼요 받은 리뷰 TOP 5
print('\n--- 도움돼요 TOP 5 리뷰 ---')
top_help = df.nlargest(5, 'helpfulCount')[['product_label','rating','helpfulCount','title','content']]
for _, row in top_help.iterrows():
    print(f"[{row['product_label']} / ★{row['rating']} / 도움:{row['helpfulCount']}] {row['title']}")
    print(f"  {str(row['content'])[:100]}...\n")

---
## 6. reviewSurveyAnswers 분석 (가성비·디자인·카메라·무게·배터리)

In [ ]:
# survey 데이터 펼치기
survey_rows = []
for _, row in df.iterrows():
    answers = row.get('reviewSurveyAnswers')
    if not isinstance(answers, list):
        continue
    for ans in answers:
        if isinstance(ans, dict) and ans.get('question') and ans.get('answer'):
            survey_rows.append({
                'product_label': row['product_label'],
                'brand': row['brand'],
                'rating': row['rating'],
                'question': ans['question'],
                'answer': ans['answer'],
            })

sv = pd.DataFrame(survey_rows)
print(f'총 survey 응답: {len(sv):,}개')
print('\n질문 종류:')
print(sv['question'].value_counts().to_string())

In [ ]:
# 질문별 답변 분포
questions = sv['question'].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, q in zip(axes, questions):
    sub = sv[sv['question'] == q]['answer'].value_counts().head(8)
    sub.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(q, fontsize=11, fontweight='bold')
    ax.set_xlabel('응답 수')
    ax.invert_yaxis()

plt.suptitle('Survey 응답 분포 (질문별)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/eda_06_survey_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 브랜드별 Survey 답변 비교 — '가성비' 항목
q_target = '가성비'
sv_q = sv[sv['question'] == q_target]

brand_answer = (
    sv_q.groupby(['brand', 'answer'])
    .size()
    .unstack(fill_value=0)
)
# 비율
brand_answer_pct = brand_answer.div(brand_answer.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 4))
brand_answer_pct.T.plot(kind='bar', ax=ax,
                        color=['#4A90D9', '#E85D5D'],
                        edgecolor='white')
ax.set_title(f'브랜드별 [{q_target}] 응답 비율', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='브랜드')
plt.tight_layout()
plt.savefig('output/eda_07_survey_brand.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 평점 그룹별 Survey 답변 — 핵심 질문들
df['rating_group'] = pd.cut(df['rating'], bins=[0,2,3,5],
                             labels=['부정(1-2)', '중립(3)', '긍정(4-5)'])

# sv에 rating_group 직접 붙이기 (index merge 대신 reviewId 기반)
sv_with_group = sv.copy()
sv_with_group['rating_group'] = sv_with_group['rating'].apply(
    lambda r: '부정(1-2)' if r <= 2 else ('중립(3)' if r == 3 else '긍정(4-5)')
)

print('\n--- 부정 리뷰(1-2점)에서 가장 많은 Survey 답변 TOP 10 ---')
neg_sv = sv[sv['rating'] <= 2]
print(neg_sv['answer'].value_counts().head(10).to_string())

print('\n--- 긍정 리뷰(4-5점)에서 가장 많은 Survey 답변 TOP 10 ---')
pos_sv = sv[sv['rating'] >= 4]
print(pos_sv['answer'].value_counts().head(10).to_string())

---
## 7. 리뷰 텍스트 기본 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 7-1. 리뷰 길이 분포
axes[0].hist(df['content_len'].clip(upper=1500), bins=50,
             color='mediumpurple', edgecolor='white')
axes[0].set_title('리뷰 길이 분포 (글자 수)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('글자 수 (최대 1500 clip)')
axes[0].axvline(df['content_len'].median(), color='red', linestyle='--',
                label=f'중앙값 {df["content_len"].median():.0f}')
axes[0].legend()

# 7-2. 평점별 리뷰 길이 분포
df.boxplot(column='content_len', by='rating', ax=axes[1],
           showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
axes[1].set_title('평점별 리뷰 길이', fontsize=11, fontweight='bold')
axes[1].set_xlabel('별점')
axes[1].set_ylabel('글자 수')
plt.suptitle('')

plt.tight_layout()
plt.savefig('output/eda_08_content_len.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('rating')['content_len'].agg(['median','mean']).round(1))

In [ ]:
# 빈 / 너무 짧은 리뷰 현황
print(f"content가 null인 리뷰: {df['content'].isna().sum()}")
print(f"content 10글자 미만: {(df['content_len'] < 10).sum()}")
print(f"content 30글자 미만: {(df['content_len'] < 30).sum()}")

print('\n--- 짧은 리뷰 예시 (10글자 미만) ---')
short = df[df['content_len'] < 10][['product_label','rating','content']].head(10)
print(short.to_string())

---
## 8. 제품별 평균 평점 · 평균 리뷰 길이 종합

In [ ]:
summary = (
    df.groupby('product_label')
    .agg(
        리뷰수=('rating', 'count'),
        평균평점=('rating', 'mean'),
        중앙평점=('rating', 'median'),
        부정비율=('rating', lambda x: (x <= 2).mean() * 100),
        평균리뷰길이=('content_len', 'mean'),
        평균도움돼요=('helpfulCount', 'mean'),
    )
    .round(2)
    .sort_values('평균평점')
)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#4A90D9' if 'iPhone' in p else '#E85D5D' for p in summary.index]
bars = ax.bar(summary.index, summary['평균평점'], color=colors, edgecolor='white')
ax.set_ylim(3.5, 5.0)
ax.set_title('제품별 평균 평점', fontsize=12, fontweight='bold')
ax.set_ylabel('평균 별점')
ax.set_xticklabels(summary.index, rotation=30, ha='right')
for bar, val in zip(bars, summary['평균평점']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('output/eda_09_avg_rating.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 이상 패턴 / 흥미로운 지점 메모

아래 셀에서 직접 발견한 패턴을 텍스트로 정리하세요.

In [ ]:
# 부정 리뷰(1-2점) 중 helpfulCount 상위 리뷰 — 공감받은 불만
neg_reviews = df[df['rating'] <= 2].nlargest(10, 'helpfulCount')
print('=== 부정 리뷰 중 도움돼요 TOP 10 ===')
for _, row in neg_reviews.iterrows():
    print(f"[{row['product_label']} ★{row['rating']} 도움:{row['helpfulCount']}] {row['title']}")
    print(f"  {str(row['content'])[:150]}")
    print()

In [ ]:
# 출시 초기(첫 달) vs 이후 리뷰의 평점 변화
df_sorted = df.sort_values('reviewAt_dt')
first_date = df_sorted['reviewAt_dt'].min()

for product in df['product_label'].unique():
    sub = df[df['product_label'] == product].sort_values('reviewAt_dt')
    if len(sub) < 20:
        continue
    cutoff = sub['reviewAt_dt'].min() + pd.Timedelta(days=30)
    early = sub[sub['reviewAt_dt'] <= cutoff]['rating'].mean()
    later = sub[sub['reviewAt_dt'] > cutoff]['rating'].mean()
    n_early = (sub['reviewAt_dt'] <= cutoff).sum()
    n_later = (sub['reviewAt_dt'] > cutoff).sum()
    print(f"{product}: 초기 {early:.2f} (n={n_early}) → 이후 {later:.2f} (n={n_later})  diff={later-early:+.2f}")

---
## 10. 다음 단계: 질문/가설 후보

EDA에서 발견한 것들을 바탕으로 아래 형식으로 정리하세요.

```
Q1. ______ 인가?
  → 왜 흥미롭나: ______
  → 어떤 방법으로 답할 수 있나: ______
  → 비즈니스 가치: ______

Q2. ...
```

힌트로 쓸 수 있는 후보 방향들:
- 삼성 vs 애플 부정 리뷰의 **불만 카테고리가 구조적으로 다른가?**
- **도움돼요를 많이 받는 리뷰**는 어떤 내용을 담고 있나? (집단 공감 신호)
- 출시 직후 vs 장기 사용자 리뷰에서 **주제가 어떻게 바뀌나?**
- Survey 응답 vs 실제 별점 — **표면 만족과 내면 불만의 gap이 있는가?**
- 특정 모델에서만 반복 등장하는 **독특한 키워드/불만**은 무엇인가?

---
---
# Topic 2. 이메일 도메인별 리뷰어 특성 분석
## "디지털 생태계 세그먼테이션: 같은 폰, 다른 시선"

> **핵심 질문**: Naver 이메일 사용자와 Gmail 사용자는 같은 스마트폰을 왜, 어떻게 다르게 평가하는가?  
> 이메일 도메인 = 디지털 라이프스타일·연령대의 **프록시 변수**로 활용

| 그룹 | 대표 도메인 | 추정 특성 |
|------|------------|----------|
| 네이버 | naver.com | 국내 생태계 중심, 중장년층 |
| Gmail | gmail.com | 글로벌/테크 친화, 젊은층 |
| 카카오 | hanmail.net, daum.net, kakao.com | 카카오 생태계, 구세대 한국 인터넷 |
| 네이트 | nate.com | SK 생태계 |
| 기타 | icloud.com, 기업 도메인 등 | 소수 집단 |

셀 1~4가 먼저 실행되어 `df`, `sv` 변수가 준비되어 있어야 합니다.

In [ ]:
import random
import numpy as np

# ── 재현성 고정 ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── 도메인 → 그룹 매핑 ──────────────────────────────────────
DOMAIN_MAP = {
    'naver.com': '네이버', 'naver.con': '네이버', 'naver.net': '네이버',
    'naver.kr': '네이버',
    'gmail.com': 'Gmail',
    'hanmail.net': '카카오', 'daum.net': '카카오', 'kakao.com': '카카오',
    'nate.com': '네이트',
}
GROUP_COLORS = {
    '네이버': '#03C75A',
    'Gmail':  '#EA4335',
    '카카오': '#F7E600',
    '네이트': '#FF6400',
    '기타':   '#AAAAAA',
}
GROUP_ORDER = ['네이버', 'Gmail', '카카오', '네이트', '기타']

def assign_group(email):
    if not isinstance(email, str) or '@' not in email:
        return '기타'
    domain = email.split('@')[-1].lower().strip()
    return DOMAIN_MAP.get(domain, '기타')

df['email_domain'] = df['member.email'].apply(
    lambda e: e.split('@')[-1].lower().strip() if isinstance(e, str) and '@' in e else 'unknown'
)
df['domain_group'] = df['member.email'].apply(assign_group)
df['attachment_count'] = df.get('attachment_count', df.apply(
    lambda _: 0, axis=1))  # fallback if not already created

# attachment_count 재계산 (안전하게)
import re, json
raw = (
    __import__('pathlib').Path('si_dataset/review_for_analysis.json')
    .read_text(encoding='utf-8')
)
raw = re.sub(r'\bNaN\b', 'null', re.sub(r'\bnullDeliberate\b', 'null', raw))
_records = json.loads(raw)
df['attachment_count'] = [len(r.get('attachments', [])) for r in _records]

print('도메인 그룹 분포:')
counts = df['domain_group'].value_counts()
for g in GROUP_ORDER:
    n = counts.get(g, 0)
    print(f'  {g:6s}: {n:5d}건 ({n/len(df)*100:.1f}%)')

### 2-1. 도메인 그룹 현황

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['domain_group'].value_counts().reindex(GROUP_ORDER)
colors_list = [GROUP_COLORS[g] for g in GROUP_ORDER]

# ── 막대 그래프 ──────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(GROUP_ORDER, counts.values, color=colors_list, edgecolor='white', linewidth=1.2)
ax.set_title('도메인 그룹별 리뷰 수', fontsize=13, fontweight='bold')
ax.set_ylabel('리뷰 수')
for bar, n in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{n:,}\n({n/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, counts.max() * 1.18)

# ── 파이 차트 ────────────────────────────────────────────────
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=GROUP_ORDER, colors=colors_list,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(linewidth=1.5, edgecolor='white'),
    textprops=dict(fontsize=10)
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('도메인 그룹 비율', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('output/t2_01_group_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# 주요 원시 도메인 확인
print('\n네이버 계열 원시 도메인:')
print(df[df['domain_group']=='네이버']['email_domain'].value_counts().to_string())
print('\n카카오 계열 원시 도메인:')
print(df[df['domain_group']=='카카오']['email_domain'].value_counts().to_string())

### 2-2. 평점 분포 비교 — Violin Plot + Kruskal-Wallis 검정

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Violin Plot ──────────────────────────────────────────────
ax = axes[0]
plot_data = [df[df['domain_group'] == g]['rating'].dropna().values for g in GROUP_ORDER]
parts = ax.violinplot(plot_data, positions=range(len(GROUP_ORDER)),
                      showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], colors_list):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_ylabel('별점')
ax.set_title('도메인 그룹별 평점 분포', fontsize=13, fontweight='bold')

# 평균 표시
for i, g in enumerate(GROUP_ORDER):
    mean_val = df[df['domain_group'] == g]['rating'].mean()
    ax.text(i, 0.5, f'평균\n{mean_val:.2f}', ha='center', va='bottom', fontsize=8,
            color='black', style='italic')

# ── 평점 1-2 (부정) 비율 ──────────────────────────────────────
ax2 = axes[1]
neg_rates = []
pos_rates = []
mean_ratings = []
for g in GROUP_ORDER:
    sub = df[df['domain_group'] == g]['rating']
    neg_rates.append((sub <= 2).mean() * 100)
    pos_rates.append((sub >= 4).mean() * 100)
    mean_ratings.append(sub.mean())

x = np.arange(len(GROUP_ORDER))
w = 0.35
bars_neg = ax2.bar(x - w/2, neg_rates, w, label='부정(1-2점) %', color='#d9534f', alpha=0.8, edgecolor='white')
bars_pos = ax2.bar(x + w/2, pos_rates, w, label='긍정(4-5점) %', color='#5cb85c', alpha=0.8, edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(GROUP_ORDER)
ax2.set_ylabel('%')
ax2.set_title('도메인 그룹별 긍·부정 리뷰 비율', fontsize=13, fontweight='bold')
ax2.legend()
for bar, v in zip(bars_neg, neg_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{v:.1f}%',
             ha='center', va='bottom', fontsize=8)
for bar, v in zip(bars_pos, pos_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{v:.1f}%',
             ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('output/t2_02_rating_violin.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Kruskal-Wallis 검정 (비모수 분산분석) ────────────────────
groups_for_test = [g for g in GROUP_ORDER if df[df['domain_group'] == g]['rating'].count() >= 10]
samples = [df[df['domain_group'] == g]['rating'].dropna().values for g in groups_for_test]
stat, p_kw = stats.kruskal(*samples)
print(f'Kruskal-Wallis H={stat:.3f}, p={p_kw:.4f}')
print('→', '그룹 간 평점 차이가 통계적으로 유의함 (p<0.05)' if p_kw < 0.05 else '그룹 간 통계적 유의 차이 없음')

# ── 사후 검정: 쌍별 Mann-Whitney U (Bonferroni) ──────────────
from itertools import combinations
print('\n[사후 검정] 쌍별 Mann-Whitney U + Bonferroni 보정')
pairs = list(combinations(groups_for_test, 2))
bonf = len(pairs)
print(f'{'그룹1':6s} vs {'그룹2':6s}  |  U-stat   p-value  보정p    유의')
print('-' * 65)
for g1, g2 in pairs:
    s1 = df[df['domain_group'] == g1]['rating'].dropna().values
    s2 = df[df['domain_group'] == g2]['rating'].dropna().values
    u, p = stats.mannwhitneyu(s1, s2, alternative='two-sided')
    p_adj = min(p * bonf, 1.0)
    sig = '★' if p_adj < 0.05 else ''
    print(f'{g1:6s} vs {g2:6s}  |  {u:8.0f}  {p:.4f}  {p_adj:.4f}  {sig}')

### 2-3. 리뷰 작성 행태 비교 (길이·사진·도움돼요)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metrics = [
    ('content_len',      '평균 리뷰 길이 (글자)',    '글자 수'),
    ('attachment_count', '평균 사진 첨부 수',         '장'),
    ('helpfulTrueCount', '평균 도움돼요 수',           '건'),
]
# 추가 지표 계산
df['has_photo'] = (df['attachment_count'] > 0).astype(int)
df['is_detailed'] = (df['content_len'] > 200).astype(int)
extra_metrics = [
    ('has_photo',    '사진 첨부 비율 (%)',    '%', True),
    ('is_detailed',  '상세 리뷰 비율 (>200자) (%)', '%', True),
]

# 상단 3개: 평균 비교 막대
for ax, (col, title, unit) in zip(axes[0], metrics):
    vals = [df[df['domain_group'] == g][col].mean() for g in GROUP_ORDER]
    bars = ax.bar(GROUP_ORDER, vals, color=colors_list, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(unit)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)

# 하단 왼쪽 2개: 비율 비교
for ax, (col, title, unit, _) in zip(axes[1][:2], extra_metrics):
    vals = [df[df['domain_group'] == g][col].mean() * 100 for g in GROUP_ORDER]
    bars = ax.bar(GROUP_ORDER, vals, color=colors_list, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(unit)
    ax.set_ylim(0, max(vals) * 1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

# 하단 오른쪽: 종합 지수 (정규화된 4개 지표 합산)
ax_last = axes[1][2]
summary_df = pd.DataFrame({
    '길이': [df[df['domain_group']==g]['content_len'].mean() for g in GROUP_ORDER],
    '사진': [df[df['domain_group']==g]['attachment_count'].mean() for g in GROUP_ORDER],
    '도움': [df[df['domain_group']==g]['helpfulTrueCount'].mean() for g in GROUP_ORDER],
    '상세비율': [df[df['domain_group']==g]['is_detailed'].mean() for g in GROUP_ORDER],
}, index=GROUP_ORDER)
# 0-1 정규화 후 합산
normalized = (summary_df - summary_df.min()) / (summary_df.max() - summary_df.min() + 1e-9)
engagement_score = normalized.mean(axis=1)
bars = ax_last.bar(GROUP_ORDER, engagement_score.values, color=colors_list, edgecolor='white', linewidth=1.2)
ax_last.set_title('리뷰 참여도 종합 지수\n(길이+사진+도움+상세 정규화 평균)', fontsize=10, fontweight='bold')
ax_last.set_ylabel('지수 (0~1)')
for bar, v in zip(bars, engagement_score.values):
    ax_last.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('도메인 그룹별 리뷰 작성 행태 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t2_03_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

# 수치 요약
print(pd.DataFrame({
    '리뷰수':     [len(df[df['domain_group']==g]) for g in GROUP_ORDER],
    '평균평점':   [round(df[df['domain_group']==g]['rating'].mean(), 2) for g in GROUP_ORDER],
    '평균길이':   [round(df[df['domain_group']==g]['content_len'].mean(), 1) for g in GROUP_ORDER],
    '사진부착률': [round(df[df['domain_group']==g]['has_photo'].mean()*100, 1) for g in GROUP_ORDER],
    '상세리뷰율': [round(df[df['domain_group']==g]['is_detailed'].mean()*100, 1) for g in GROUP_ORDER],
    '평균도움':   [round(df[df['domain_group']==g]['helpfulTrueCount'].mean(), 2) for g in GROUP_ORDER],
}, index=GROUP_ORDER).to_string())

### 2-4. 브랜드·제품 선호도 히트맵

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── 브랜드 선호도 (Apple vs Samsung 비율) ────────────────────
brand_group = (
    df.groupby(['domain_group', 'brand'])
    .size().unstack(fill_value=0)
    .reindex(GROUP_ORDER)
)
brand_pct = brand_group.div(brand_group.sum(axis=1), axis=0) * 100

ax = axes[0]
brand_pct.plot(kind='bar', ax=ax, color=['#4A90D9', '#E85D5D'],
               edgecolor='white', width=0.65)
ax.set_title('도메인 그룹별 브랜드 선호 비율', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='브랜드')
ax.set_ylim(0, 115)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)

# ── 제품별 구매 분포 히트맵 ───────────────────────────────────
product_group = (
    df.groupby(['domain_group', 'product_label'])
    .size().unstack(fill_value=0)
    .reindex(GROUP_ORDER)
)
product_pct = product_group.div(product_group.sum(axis=1), axis=0) * 100

ax2 = axes[1]
sns.heatmap(product_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax2, cbar_kws={'label': '%'})
ax2.set_title('도메인 그룹별 구매 제품 분포 (%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('output/t2_04_brand_product.png', dpi=150, bbox_inches='tight')
plt.show()

# 수치 출력
print('\n[브랜드 선호 비율]')
print(brand_pct.round(1).to_string())
print('\n[제품별 구매 비율]')
print(product_pct.round(1).to_string())

### 2-5. Survey 응답 패턴 — 레이더 차트 (그룹별 가치 우선순위)

In [ ]:
import math

# ── Survey 데이터에 domain_group 합치기 ──────────────────────
survey_rows2 = []
for _, row in df.iterrows():
    answers = row.get('reviewSurveyAnswers')
    if not isinstance(answers, list):
        continue
    for ans in answers:
        if isinstance(ans, dict) and ans.get('question') and ans.get('answer'):
            survey_rows2.append({
                'domain_group': row['domain_group'],
                'rating': row['rating'],
                'question': ans['question'],
                'answer': ans['answer'],
            })
sv2 = pd.DataFrame(survey_rows2)

# ── 긍정 점수 매핑 (답변 텍스트 키워드 기반) ─────────────────
POSITIVE_KW  = ['아주만족', '아주뛰어', '충분', '가볍', '마음에', '좋아', '뛰어나', '만족']
NEGATIVE_KW  = ['별로', '짧아', '비싸', '무거', '안좋', '불편', '짧은']

def sentiment_score(answer: str) -> float:
    a = str(answer).replace(' ', '')
    for kw in POSITIVE_KW:
        if kw in a:
            return 1.0
    for kw in NEGATIVE_KW:
        if kw in a:
            return 0.0
    return 0.5   # 적당, 보통 등 중립

sv2['score'] = sv2['answer'].apply(sentiment_score)

# ── 질문별 그룹 평균 긍정 점수 ───────────────────────────────
# 5개 핵심 질문만 사용 (샘플 적은 질문 제외)
TOP_QUESTIONS = sv2['question'].value_counts().head(5).index.tolist()
radar_df = (
    sv2[sv2['question'].isin(TOP_QUESTIONS)]
    .groupby(['domain_group', 'question'])['score']
    .mean()
    .unstack(fill_value=0.5)
    .reindex(GROUP_ORDER)
    [TOP_QUESTIONS]
)
print('레이더 데이터:')
print(radar_df.round(3).to_string())

# ── 레이더 차트 ───────────────────────────────────────────────
categories = TOP_QUESTIONS
N = len(categories)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]  # 닫기

fig, axes_radar = plt.subplots(1, 2, figsize=(16, 7),
                                subplot_kw=dict(projection='polar'))

# 왼쪽: 4개 주요 그룹 겹쳐 그리기
ax_r = axes_radar[0]
main_groups = ['네이버', 'Gmail', '카카오', '네이트']
for g in main_groups:
    if g not in radar_df.index:
        continue
    values = radar_df.loc[g].tolist()
    values += values[:1]
    ax_r.plot(angles, values, linewidth=2, label=g, color=GROUP_COLORS[g])
    ax_r.fill(angles, values, alpha=0.12, color=GROUP_COLORS[g])

ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(categories, fontsize=10)
ax_r.set_ylim(0, 1)
ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_r.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
ax_r.set_title('도메인 그룹별 만족도 레이더\n(0=부정, 0.5=중립, 1=긍정)', fontsize=11, fontweight='bold', pad=20)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# 오른쪽: 질문별 그룹 비교 막대 (레이더 보조)
ax2 = axes_radar[1]
ax2.set_visible(False)  # polar subplot 숨기고 새 axes로 대체
fig.add_axes([0.55, 0.1, 0.42, 0.8])
ax_bar = fig.axes[-1]

x = np.arange(len(TOP_QUESTIONS))
bar_w = 0.18
for i, g in enumerate(main_groups):
    if g not in radar_df.index:
        continue
    vals = radar_df.loc[g].values
    ax_bar.bar(x + i * bar_w, vals, bar_w, label=g,
               color=GROUP_COLORS[g], alpha=0.85, edgecolor='white')

ax_bar.set_xticks(x + bar_w * 1.5)
ax_bar.set_xticklabels(TOP_QUESTIONS, rotation=20, ha='right', fontsize=9)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_ylabel('긍정 점수 (0~1)')
ax_bar.set_title('질문별 그룹 긍정 점수 비교', fontsize=11, fontweight='bold')
ax_bar.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax_bar.legend(fontsize=8, loc='upper right')
ax_bar.set_facecolor('#f8f8f8')

plt.savefig('output/t2_05_radar_survey.png', dpi=150, bbox_inches='tight')
plt.show()

### 2-6. 도메인 그룹별 핵심 키워드 — TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

# ── 불용어 (한국어 기능어 및 노이즈) ────────────────────────
STOPWORDS = {
    '이', '가', '을', '를', '은', '는', '의', '에', '에서', '로', '으로', '도',
    '하다', '이다', '있다', '없다', '하고', '하여', '해서', '하면', '것', '수',
    '더', '또', '그', '이', '저', '제', '내', '그리고', '하지만', '그런데',
    '정말', '너무', '아주', '매우', '좀', '조금', '많이', '많은', '아이폰',
    '갤럭시', '구매', '제품', '사용', '배송', '리뷰', '후기', '구입', '전화기',
    '스마트폰', '폰', '핸드폰', '폰이', '폰을', '폰의',
}

def tokenize_ko(text: str) -> str:
    """공백 기준 분리 후 2글자 이상 토큰만 반환 (불용어 제거)"""
    tokens = [t for t in str(text).split() if len(t) >= 2 and t not in STOPWORDS]
    return ' '.join(tokens)

# ── 그룹별 문서 합치기 ────────────────────────────────────────
main_groups_tfidf = ['네이버', 'Gmail', '카카오', '네이트']
group_corpus = {}
for g in main_groups_tfidf:
    texts = df[df['domain_group'] == g]['content'].fillna('').tolist()
    combined = ' '.join([tokenize_ko(t) for t in texts])
    group_corpus[g] = combined

# ── TF-IDF (그룹을 문서로) ────────────────────────────────────
vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),   # 단어 + 2-gram
    max_features=3000,
    min_df=1,
    sublinear_tf=True,
)
tfidf_matrix = vectorizer.fit_transform(list(group_corpus.values()))
feature_names = vectorizer.get_feature_names_out()

# 그룹별 상위 키워드 추출
TOP_N = 20
group_keywords = {}
for i, g in enumerate(main_groups_tfidf):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:TOP_N]
    group_keywords[g] = [(feature_names[j], scores[j]) for j in top_idx]

# 출력
for g, kws in group_keywords.items():
    print(f'\n[{g}] 상위 {TOP_N} 키워드:')
    print('  ' + ', '.join([f'{kw}({sc:.3f})' for kw, sc in kws]))

In [ ]:
# ── 워드클라우드 (2×2 그리드) ────────────────────────────────
FONT_PATH = r'C:\Windows\Fonts\malgun.ttf'

fig, axes_wc = plt.subplots(2, 2, figsize=(16, 12))
axes_wc = axes_wc.flatten()

for ax, g in zip(axes_wc, main_groups_tfidf):
    kw_dict = {kw: sc for kw, sc in group_keywords[g]}
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color='white',
        width=600, height=400,
        max_words=60,
        colormap='Set2',
        random_state=SEED,
    ).generate_from_frequencies(kw_dict)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'[{g}] 핵심 키워드', fontsize=13, fontweight='bold',
                 color=GROUP_COLORS[g], pad=10)

plt.suptitle('도메인 그룹별 핵심 키워드 워드클라우드 (TF-IDF)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t2_06_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

### 2-7. 구매 시점 분포 — 도메인 그룹별 얼리어답터 비율

In [ ]:
# 출시일 정의 (공식 발표일 기준)
RELEASE_DATES = {
    'iphone_17':          pd.Timestamp('2025-09-19'),
    'iphone_17_pro':      pd.Timestamp('2025-09-19'),
    'iphone_17_pro_max':  pd.Timestamp('2025-09-19'),
    'galaxy_s26':         pd.Timestamp('2026-01-22'),
    'galaxy_s26_ultra':   pd.Timestamp('2026-01-22'),
    'galaxy_z_fold7':     pd.Timestamp('2026-07-09'),
    'galaxy_z_flip7':     pd.Timestamp('2026-07-09'),
}

df['release_date'] = df['product_name'].map(RELEASE_DATES)
df['days_since_release'] = (df['reviewAt_dt'] - df['release_date']).dt.days

# 코호트 정의
def cohort(days):
    if pd.isna(days) or days < 0:
        return '출시 전'
    elif days <= 30:
        return 'D+0~30 (얼리어답터)'
    elif days <= 90:
        return 'D+31~90 (초기다수)'
    else:
        return 'D+91+ (후기다수)'

df['purchase_cohort'] = df['days_since_release'].apply(cohort)

# ── 도메인 그룹별 코호트 분포 ────────────────────────────────
cohort_order = ['D+0~30 (얼리어답터)', 'D+31~90 (초기다수)', 'D+91+ (후기다수)']
cohort_group = (
    df[df['domain_group'].isin(main_groups_tfidf)]
    .groupby(['domain_group', 'purchase_cohort'])
    .size().unstack(fill_value=0)
    .reindex(main_groups_tfidf)
    .reindex(columns=[c for c in cohort_order if c in df['purchase_cohort'].unique()], fill_value=0)
)
cohort_pct = cohort_group.div(cohort_group.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 스택 막대
cohort_pct.plot(kind='bar', stacked=True, ax=axes[0],
                color=['#2196F3', '#FF9800', '#9C27B0'],
                edgecolor='white', width=0.6)
axes[0].set_title('도메인 그룹별 구매 코호트 분포', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right', fontsize=8)

# 얼리어답터 비율만 강조
early_rate = cohort_pct.get('D+0~30 (얼리어답터)', pd.Series(dtype=float))
if len(early_rate):
    bars = axes[1].bar(early_rate.index,
                       early_rate.values,
                       color=[GROUP_COLORS[g] for g in early_rate.index],
                       edgecolor='white', linewidth=1.2)
    axes[1].set_title('얼리어답터 비율 (D+0~30)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('%')
    axes[1].tick_params(axis='x', rotation=0)
    for bar, v in zip(bars, early_rate.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/t2_07_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n[코호트별 평균 평점 (그룹 × 코호트)]')
rating_cohort = (
    df[df['domain_group'].isin(main_groups_tfidf)]
    .groupby(['domain_group', 'purchase_cohort'])['rating']
    .mean().unstack().round(2)
)
print(rating_cohort.to_string())

### 2-8. 종합 인사이트 요약

In [ ]:
print('=' * 70)
print('Topic 2 — 종합 인사이트 요약')
print('=' * 70)

for g in main_groups_tfidf:
    sub = df[df['domain_group'] == g]
    print(f'\n▶ [{g}] (n={len(sub)})')
    print(f'   평균 평점   : {sub["rating"].mean():.2f}  '
          f'  부정비율 {(sub["rating"]<=2).mean()*100:.1f}%')
    print(f'   Apple 비율  : {(sub["brand"]=="Apple").mean()*100:.1f}%  '
          f'  Samsung 비율 {(sub["brand"]=="Samsung").mean()*100:.1f}%')
    print(f'   평균 리뷰길이: {sub["content_len"].mean():.0f}자  '
          f'  사진부착률 {sub["has_photo"].mean()*100:.1f}%')
    # 가장 많이 구매한 제품
    top_product = sub['product_label'].mode().iloc[0] if len(sub) else '-'
    print(f'   인기 제품   : {top_product}')
    # 얼리어답터 비율
    early = (sub['purchase_cohort'] == 'D+0~30 (얼리어답터)').mean() * 100
    print(f'   얼리어답터  : {early:.1f}%')

print('\n' + '=' * 70)
print('비즈니스 가치 연결')
print('=' * 70)
print("""
① 마케팅 채널 전략
   - 네이버 유저 다수 → 네이버쇼핑/스마트스토어 광고 집중
   - Gmail 유저 → 구글 광고, 유튜브 제품 리뷰 타겟팅

② 콘텐츠 차별화
   - 그룹별 TF-IDF 키워드 → 각 채널 광고 카피로 활용
   - (예) 네이버 유저는 '배송/AS/가성비' 강조 / Gmail은 '성능/카메라/디자인'

③ 리뷰 플랫폼 신뢰성
   - 참여도 종합 지수가 높은 그룹의 리뷰 = 양질의 UGC
   - 플랫폼이 해당 그룹 리뷰를 상단 노출 시 구매전환율 ↑ 기대

④ 브랜드 선호도 차이
   - 도메인별 Apple/Samsung 구매 비율 차이
     → 삼성 = 네이버/카카오 생태계 친화, 애플 = Gmail 친화 가설 검증
""")


---
### 2-13. 동일 제품 × 도메인 그룹 평점 비교
> "같은 폰을 샀는데 왜 다르게 평가하나?" — 제품 통제 후 그룹 간 평점 차이 검증

In [ ]:
# ── 제품 × 도메인 평균 평점 피벗 ─────────────────────────────
focus_groups = ['네이버', 'Gmail', '카카오', '네이트']
pivot = (
    df[df['domain_group'].isin(focus_groups)]
    .groupby(['product_label', 'domain_group'])['rating']
    .mean()
    .unstack()
    .reindex(columns=focus_groups)
    .round(2)
)
# 전체 평균 대비 차이 (delta)
overall_mean = df.groupby('product_label')['rating'].mean()
pivot_delta = pivot.subtract(overall_mean, axis=0).round(2)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── 절대 평점 히트맵 ──────────────────────────────────────────
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=3.8, vmax=5.0, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '평균 평점'})
axes[0].set_title('제품 × 도메인 평균 평점', fontsize=12, fontweight='bold')
axes[0].set_ylabel('제품')
axes[0].set_xlabel('')

# ── 전체 평균 대비 delta 히트맵 ───────────────────────────────
sns.heatmap(pivot_delta, annot=True, fmt='+.2f', cmap='RdBu',
            center=0, linewidths=0.5, ax=axes[1],
            cbar_kws={'label': '전체 평균 대비 Δ'})
axes[1].set_title('전체 평균 대비 편차\n(+ = 전체보다 후한 평가)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')
axes[1].set_xlabel('')

# ── 네이버 vs Gmail 차이 (제품별) ────────────────────────────
naver_gmail_diff = (pivot['Gmail'] - pivot['네이버']).dropna().sort_values()
colors_diff = ['#4A90D9' if v > 0 else '#E85D5D' for v in naver_gmail_diff]
bars = axes[2].barh(naver_gmail_diff.index, naver_gmail_diff.values,
                    color=colors_diff, edgecolor='white', linewidth=1.2)
axes[2].axvline(0, color='gray', linewidth=1, linestyle='--')
axes[2].set_title('Gmail - 네이버 평점 차이\n(+ = Gmail이 더 후하게 평가)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('평점 차이')
for bar, v in zip(bars, naver_gmail_diff.values):
    axes[2].text(v + (0.005 if v >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
                 f'{v:+.2f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('output/t2_13_product_group_rating.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 제품 통제 후 Kruskal-Wallis (각 제품 내에서 그룹 차이) ──
print('=== 제품 통제 후 그룹 간 평점 차이 검증 (Kruskal-Wallis) ===')
print(f'{"제품":<22}  H-stat    p-value  유의')
print('-' * 50)
for product in df['product_label'].dropna().unique():
    sub = df[(df['product_label'] == product) & (df['domain_group'].isin(focus_groups))]
    samples = [sub[sub['domain_group'] == g]['rating'].dropna().values
               for g in focus_groups if len(sub[sub['domain_group'] == g]) >= 5]
    if len(samples) < 2:
        continue
    stat, p = stats.kruskal(*samples)
    sig = '★★' if p < 0.01 else ('★' if p < 0.05 else '')
    print(f'{product:<22}  {stat:6.2f}  {p:.4f}  {sig}')

print('\n★★ p<0.01, ★ p<0.05')
print('\n[피벗 수치]')
print(pivot.to_string())

### 2-14. 리뷰 영향력(Helpfulness) 회귀 분석
> "어떤 그룹이 쓴 리뷰가 더 많은 공감을 받나?" — 플랫폼 리뷰 노출 알고리즘 개선 근거

In [ ]:
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler

# ── 회귀 데이터 준비 ──────────────────────────────────────────
reg_df = df[df['domain_group'].isin(focus_groups)].copy()
reg_df = reg_df.dropna(subset=['helpfulTrueCount', 'content_len', 'rating', 'domain_group'])

# 특성 정의
reg_df['log_len']    = np.log1p(reg_df['content_len'])
reg_df['is_neg']     = (reg_df['rating'] <= 2).astype(int)
reg_df['is_pos']     = (reg_df['rating'] >= 4).astype(int)
reg_df['has_photo']  = (reg_df['attachment_count'] > 0).astype(int)
reg_df['is_samsung'] = (reg_df['brand'] == 'Samsung').astype(int)

# 도메인 더미 (네이버 기준)
for g in ['Gmail', '카카오', '네이트']:
    reg_df[f'grp_{g}'] = (reg_df['domain_group'] == g).astype(int)

feature_cols = ['log_len', 'has_photo', 'is_neg', 'is_pos',
                'is_samsung', 'grp_Gmail', 'grp_카카오', 'grp_네이트']
X = reg_df[feature_cols].values
y = reg_df['helpfulTrueCount'].values.astype(float)

# Poisson Regression (도움돼요 = count data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

poisson_reg = PoissonRegressor(alpha=0.1, max_iter=300)
poisson_reg.fit(X_scaled, y)
coefs = dict(zip(feature_cols, poisson_reg.coef_))

# ── 도메인 그룹별 평균 helpfulness (통제 전/후) ──────────────
print('=== 도메인 그룹별 helpfulTrueCount 비교 ===')
print(f'{"그룹":<8}  평균  중앙값  n')
for g in focus_groups:
    sub = reg_df[reg_df['domain_group'] == g]['helpfulTrueCount']
    print(f'{g:<8}  {sub.mean():.2f}   {sub.median():.0f}    {len(sub)}')

# ── 계수 시각화 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 회귀 계수 (특성별 helpfulness 영향)
coef_series = pd.Series(coefs).sort_values()
colors_coef = ['#5cb85c' if v > 0 else '#d9534f' for v in coef_series.values]
bars_c = axes[0].barh(coef_series.index, coef_series.values,
                       color=colors_coef, edgecolor='white', linewidth=1.2)
axes[0].axvline(0, color='gray', linewidth=1, linestyle='--')
axes[0].set_title('Poisson 회귀 계수\n(도움돼요 수에 대한 영향, 기준=네이버)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('계수 (exp 변환 전, 크기 = 영향 강도)')

# 가독성 있는 라벨
label_map = {
    'log_len': '리뷰 길이 (log)',
    'has_photo': '사진 첨부',
    'is_neg': '부정 리뷰 (1-2점)',
    'is_pos': '긍정 리뷰 (4-5점)',
    'is_samsung': '삼성 제품',
    'grp_Gmail': 'Gmail 그룹',
    'grp_카카오': '카카오 그룹',
    'grp_네이트': '네이트 그룹',
}
axes[0].set_yticklabels([label_map.get(l, l) for l in coef_series.index])
for bar, v in zip(bars_c, coef_series.values):
    axes[0].text(v + (0.005 if v >= 0 else -0.005),
                 bar.get_y() + bar.get_height()/2,
                 f'{v:+.3f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=9)

# 그룹별 helpfulness 분포 (box + jitter)
help_data = [reg_df[reg_df['domain_group'] == g]['helpfulTrueCount'].clip(upper=30).values
             for g in focus_groups]
bp = axes[1].boxplot(help_data, labels=focus_groups, patch_artist=True,
                      showfliers=False,
                      medianprops=dict(color='black', linewidth=2))
for patch, g in zip(bp['boxes'], focus_groups):
    patch.set_facecolor(GROUP_COLORS[g])
    patch.set_alpha(0.7)

# jitter
for i, (g, data) in enumerate(zip(focus_groups, help_data)):
    jitter = np.random.normal(i + 1, 0.08, size=min(len(data), 200))
    sample_idx = np.random.choice(len(data), min(len(data), 200), replace=False)
    axes[1].scatter(jitter, data[sample_idx], alpha=0.3, s=8, color=GROUP_COLORS[g])

axes[1].set_title('그룹별 helpfulTrueCount 분포\n(최대 30 clip, jitter 샘플)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('도움돼요 수')

plt.tight_layout()
plt.savefig('output/t2_14_helpfulness_reg.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== 회귀 계수 해석 (IRR = exp(coef)) ===')
print(f'{"변수":<18}  계수      IRR      해석')
print('-' * 65)
for feat, coef in sorted(coefs.items(), key=lambda x: -abs(x[1])):
    irr = np.exp(coef)
    lbl = label_map.get(feat, feat)
    direction = '↑ 도움 증가' if irr > 1 else '↓ 도움 감소'
    print(f'{lbl:<18}  {coef:+.3f}  {irr:.3f}x  {direction}')

### 2-15. 그룹별 대표 리뷰 & 최종 인사이트 내러티브
> 수치 뒤에 있는 실제 목소리 — 각 그룹을 대표하는 리뷰 추출

In [ ]:
print('=' * 70)
print('그룹별 대표 리뷰 — "도움돼요" 상위 + 평점 1-2점 각 2건')
print('=' * 70)

for g in focus_groups:
    sub = df[df['domain_group'] == g]
    print(f'\n{"─"*60}')
    print(f'▶ [{g}] 그룹  (총 {len(sub)}건 | 평균 평점 {sub["rating"].mean():.2f})')
    print(f'{"─"*60}')

    # 가장 많은 도움돼요를 받은 리뷰
    top_help = sub.nlargest(2, 'helpfulTrueCount')[
        ['rating', 'helpfulTrueCount', 'product_label', 'content']
    ]
    print(f'\n  [👍 도움돼요 상위 리뷰]')
    for _, row in top_help.iterrows():
        preview = str(row['content'])[:200].replace('\n', ' ')
        print(f'  ★{row["rating"]} | 도움:{row["helpfulTrueCount"]} | {row["product_label"]}')
        print(f'  "{preview}..."')
        print()

    # 부정 리뷰 중 도움돼요가 높은 것
    neg_top = sub[sub['rating'] <= 2].nlargest(1, 'helpfulTrueCount')[
        ['rating', 'helpfulTrueCount', 'product_label', 'content']
    ]
    if len(neg_top):
        print(f'  [😤 부정 리뷰 중 공감 1위]')
        for _, row in neg_top.iterrows():
            preview = str(row['content'])[:200].replace('\n', ' ')
            print(f'  ★{row["rating"]} | 도움:{row["helpfulTrueCount"]} | {row["product_label"]}')
            print(f'  "{preview}..."')
            print()

In [ ]:
print('=' * 70)
print('Topic 2 — 최종 비즈니스 인사이트 종합')
print('=' * 70)
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 1. 이메일 도메인 = 가격 민감도 대리 변수
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 동일 제품에서도 그룹별 평점 차이 존재 → Kruskal-Wallis 유의 제품 확인
• Gmail 유저: A/S 불만(21%) & 배터리 불만 높음 → 고관여 소비자, 기대치 높음
• 카카오 유저: 평균 평점 4.70 (최고), 가성비 불만 낮음 → 수용도 높은 세그먼트
• 비즈니스 제안: 동일 쿠폰/할인 대신 세그먼트별 다른 메시지
  - 네이버 → "배송 안심 보장 + 가성비" 강조
  - Gmail → "성능 보증 + A/S 1년 연장" 강조
  - 카카오 → 감성 디자인 콘텐츠 중심

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 2. 부정 리뷰의 공감 구조 차이 → 리뷰 표시 알고리즘 개선 기회
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Poisson 회귀: 부정 리뷰가 긍정 리뷰보다 helpfulness에 더 큰 영향
• 그룹 계수: Gmail > 네이트 > 카카오 순 (네이버 기준 상대값)
  → Gmail 유저가 쓴 리뷰가 내용 통제 후에도 더 많은 공감 수집
• 비즈니스 제안 (쿠팡):
  - 구매 결정 단계에서 Gmail 그룹 리뷰를 상단 노출 → 전환율 ↑ 기대
  - 부정 리뷰 중 Gmail 발 A/S 불만이 가장 공감 높음 → 제조사에 품질 피드백

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 3. 디지털 생태계 ≠ 브랜드 충성도 (예상과 다른 결과)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 가설: "Gmail 유저 = 애플 선호" → 실제 데이터: 모든 그룹에서 삼성 비율이 더 높음
• Gmail도 삼성 57.8% vs 애플 42.2% → "구글 생태계 = 삼성 생태계 중복 사용자" 다수
• BERTopic T5(프로맥스 비교 리뷰): Gmail 그룹에 집중 → 프리미엄 모델 비교 구매자
• 비즈니스 제안: 삼성의 구글 서비스 연동 강화가 Gmail 유저 유지에 효과적
""")

# 샘플 사이즈 한계 안내
print('⚠️  한계: iCloud 그룹(n=7) 제외, 카카오/네이트는 n<300으로 통계적 해석 주의')

---
---
# Topic 3. 출시 시점-구매 시점 차이 → 리뷰 라이프사이클 분석
## "같은 폰, 다른 시간: 얼리어답터와 일반 구매자는 같은 폰을 다르게 경험하는가?"

**핵심 질문**
1. 출시 직후 구매자(얼리어답터)와 수개월 후 구매자(일반 소비자)의 평점이 다른가?
2. 리뷰의 관심사·키워드는 출시 후 시간이 지남에 따라 어떻게 바뀌는가?
3. 초기 리뷰가 플랫폼 전체 평점을 왜곡하는가? (허니문 효과)
4. 출시 초기 리뷰가 더 많은 공감을 받는가?

**코호트 정의**

| 코호트 | 기준 | 소비자 유형 |
|--------|------|-----------|
| D+0~30 | 출시 후 30일 이내 | 얼리어답터 — 브랜드 충성, 높은 관여도 |
| D+31~90 | 출시 31~90일 | 초기 다수 — 입소문 듣고 구매 |
| D+91+ | 출시 91일 이후 | 후기 다수 — 가격 하락 후 구매, 장기 사용자 |

셀 1~4 + Topic 2 셀이 먼저 실행되어 있어야 합니다 (`df`, `kiwi`, `SEED` 등 변수 필요).

In [ ]:
# ── 기준점: 제품별 첫 리뷰 날짜 (데이터 내 최솟값) ──────────
# 공식 출시일 대신 실제 데이터에서 가장 빠른 리뷰를 D+0으로 사용
RELEASE_DATES_T3 = df.groupby('product_name')['reviewAt_dt'].min().to_dict()

print('제품별 D+0 기준점 (첫 리뷰 날짜):')
_pmap = {
    'iphone_17': 'iPhone 17', 'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26', 'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7', 'galaxy_z_flip7': 'Galaxy Z Flip7',
}
for prod, dt in sorted(RELEASE_DATES_T3.items()):
    print(f'  {_pmap.get(prod, prod):<22}: {dt.strftime("%Y-%m-%d")}')

PRODUCT_LABELS = {
    'iphone_17': 'iPhone 17', 'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26', 'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7', 'galaxy_z_flip7': 'Galaxy Z Flip7',
}
BRAND_COLORS_T3 = {
    'iPhone 17': '#4A90D9', 'iPhone 17 Pro': '#2C5F8A', 'iPhone 17 Pro Max': '#1A3A55',
    'Galaxy S26': '#E85D5D', 'Galaxy S26 Ultra': '#A52020',
    'Galaxy Z Fold7': '#E8943A', 'Galaxy Z Flip7': '#B85E10',
}
COHORT_ORDER  = ['D+0~30 (얼리어답터)', 'D+31~90 (초기다수)', 'D+91+ (후기다수)']
COHORT_COLORS = {'D+0~30 (얼리어답터)': '#2196F3',
                 'D+31~90 (초기다수)':  '#FF9800',
                 'D+91+ (후기다수)':    '#9C27B0'}

# ── days_since_first_review 계산 ─────────────────────────────
df['release_date_t3'] = df['product_name'].map(RELEASE_DATES_T3)
df['days_t3'] = (df['reviewAt_dt'] - df['release_date_t3']).dt.days

def assign_cohort(d):
    if pd.isna(d) or d < 0: return '출시 전'
    if d <= 30:  return 'D+0~30 (얼리어답터)'
    if d <= 90:  return 'D+31~90 (초기다수)'
    return 'D+91+ (후기다수)'

df['cohort'] = df['days_t3'].apply(assign_cohort)

# 코호트 분포 확인
print('
코호트 분포:')
cohort_dist = df[df['cohort'] != '출시 전'].groupby(['product_label', 'cohort']).size().unstack(fill_value=0)
cohort_dist = cohort_dist.reindex(columns=[c for c in COHORT_ORDER if c in cohort_dist.columns])
print(cohort_dist.to_string())
print(f'
전체: {(df["cohort"]!="출시 전").sum()}건 / 출시 전: {(df["cohort"]=="출시 전").sum()}건')


### 3-1. 리뷰 볼륨 타임라인 — 언제 리뷰가 폭발하는가

In [ ]:
df_valid = df[df['cohort'] != '출시 전'].copy()

# 10일 bin으로 집계
df_valid['days_bin'] = (df_valid['days_t3'] // 10) * 10

products = list(PRODUCT_LABELS.values())
# Apple vs Samsung 나눠서 2행 4열
apple_prods = [p for p in products if 'iPhone' in p]
sam_prods   = [p for p in products if 'Galaxy' in p]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax_idx, prod in enumerate(apple_prods + sam_prods):
    ax = axes[ax_idx]
    pname = [k for k, v in PRODUCT_LABELS.items() if v == prod][0]
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty:
        ax.axis('off'); continue

    vol = sub.groupby('days_bin').size().reset_index(name='count')
    color = BRAND_COLORS_T3[prod]

    bars = ax.bar(vol['days_bin'], vol['count'], width=9,
                  color=color, alpha=0.8, edgecolor='white')
    # 최고 스파이크 강조
    peak_day = vol.loc[vol['count'].idxmax(), 'days_bin']
    peak_cnt = vol['count'].max()
    ax.axvline(peak_day, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.text(peak_day + 1, peak_cnt * 0.95,
            f'Peak\nD+{int(peak_day)}~{int(peak_day)+9}', fontsize=7, color='red')

    # 코호트 경계선
    for boundary, label in [(30, 'D+30'), (90, 'D+90')]:
        if boundary <= sub['days_t3'].max():
            ax.axvline(boundary, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            ax.text(boundary + 1, ax.get_ylim()[1] * 0.85, label, fontsize=7, color='gray')

    ax.set_title(prod, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('출시 후 경과 일수 (10일 bin)')
    ax.set_ylabel('리뷰 수')

for ax in axes[len(apple_prods + sam_prods):]:
    ax.axis('off')

plt.suptitle('제품별 리뷰 볼륨 타임라인 (출시 후 경과 일수 기준)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_01_volume_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# 요약
print('제품별 리뷰 볼륨 피크 시점:')
for prod in apple_prods + sam_prods:
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty: continue
    vol = sub.groupby('days_bin').size()
    peak = vol.idxmax()
    total_days = sub['days_t3'].max()
    print(f'  {prod:<22} 피크: D+{int(peak)}~{int(peak)+9}일  '
          f'(전체 관찰기간: D+0~{int(total_days)})')

### 3-2. 평점 시계열 + 허니문 효과 검증

In [ ]:
# ── 평점 시계열 (10일 bin 이동평균) ───────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax_idx, prod in enumerate(apple_prods + sam_prods):
    ax = axes[ax_idx]
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty: ax.axis('off'); continue

    color = BRAND_COLORS_T3[prod]
    rating_by_bin = sub.groupby('days_bin')['rating'].agg(['mean', 'count', 'sem']).reset_index()
    rating_by_bin = rating_by_bin[rating_by_bin['count'] >= 3]

    # 산점도 (개별 리뷰, jitter)
    jitter = np.random.normal(0, 1.5, len(sub))
    ax.scatter(sub['days_t3'] + jitter, sub['rating'],
               alpha=0.05, s=5, color=color)

    # 10일 bin 평균 + 95% CI
    ax.plot(rating_by_bin['days_bin'], rating_by_bin['mean'],
            color=color, linewidth=2.5, zorder=5)
    ax.fill_between(rating_by_bin['days_bin'],
                    rating_by_bin['mean'] - 1.96 * rating_by_bin['sem'],
                    rating_by_bin['mean'] + 1.96 * rating_by_bin['sem'],
                    alpha=0.2, color=color, label='95% CI')

    # 코호트 경계
    for b in [30, 90]:
        if b < sub['days_t3'].max():
            ax.axvline(b, color='gray', linestyle=':', linewidth=1)

    ax.set_ylim(1, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_title(prod, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('출시 후 경과 일수')
    ax.set_ylabel('평점')
    ax.axhline(sub['rating'].mean(), color='black', linestyle='--',
               linewidth=0.8, alpha=0.5)

for ax in axes[len(apple_prods + sam_prods):]:
    ax.axis('off')

plt.suptitle('제품별 평점 시계열 (출시 후 경과 일수 기준)\n점선=전체 평균, 회색점선=코호트 경계',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_02_rating_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 허니문 효과 검증: 코호트별 평점 + Mann-Whitney U ──────────
cohort_rating = (
    df_valid[df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby(['product_label', 'cohort'])['rating']
    .mean().unstack().reindex(columns=COHORT_ORDER)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 코호트별 평균 평점 히트맵
sns.heatmap(cohort_rating, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=3.8, vmax=5.0, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '평균 평점'})
axes[0].set_title('제품 × 코호트 평균 평점\n(허니문 효과 = 얼리어답터 > 후기다수)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('')

# 얼리어답터 vs 후기다수 차이 막대
early_col = 'D+0~30 (얼리어답터)'
late_col  = 'D+91+ (후기다수)'
diff_df = cohort_rating[[early_col, late_col]].dropna()
diff = (diff_df[early_col] - diff_df[late_col]).sort_values()

bar_colors = ['#d9534f' if v > 0 else '#5cb85c' for v in diff.values]
bars = axes[1].barh(diff.index, diff.values, color=bar_colors, edgecolor='white', linewidth=1.2)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('얼리어답터 − 후기다수 평점 차이\n(빨강=얼리어답터가 더 후한 평가, 초록=반대)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('평점 차이')
for bar, v in zip(bars, diff.values):
    axes[1].text(v + (0.005 if v >= 0 else -0.005),
                 bar.get_y() + bar.get_height()/2,
                 f'{v:+.2f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/t3_03_honeymoon.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Mann-Whitney U (얼리어답터 vs 후기다수, 제품별) ──────────
print('=== 허니문 효과 통계 검증 (얼리어답터 D+0~30 vs 후기다수 D+91+) ===')
print(f'{"제품":<22}  조기평균  후기평균  차이    U-stat   p-value  유의')
print('-' * 75)
honeymoon_products = []
for prod in apple_prods + sam_prods:
    sub = df_valid[df_valid['product_label'] == prod]
    early = sub[sub['cohort'] == early_col]['rating'].dropna()
    late  = sub[sub['cohort'] == late_col]['rating'].dropna()
    if len(early) < 5 or len(late) < 5:
        print(f'{prod:<22}  (샘플 부족)')
        continue
    u, p = stats.mannwhitneyu(early, late, alternative='two-sided')
    diff_val = early.mean() - late.mean()
    sig = '★★' if p < 0.01 else ('★' if p < 0.05 else '')
    if diff_val > 0 and p < 0.05:
        honeymoon_products.append(prod)
    print(f'{prod:<22}  {early.mean():.2f}    {late.mean():.2f}    '
          f'{diff_val:+.2f}   {u:8.0f}  {p:.4f}  {sig}')

print(f'\n허니문 효과 유의 제품: {honeymoon_products if honeymoon_products else "없음"}')

### 3-3. Helpfulness 시계열 — 초기 리뷰가 더 많은 공감을 받는가

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 코호트별 평균 helpfulness ────────────────────────────────
cohort_help = (
    df_valid[df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby('cohort')['helpfulTrueCount']
    .agg(['mean', 'median', 'count'])
    .reindex(COHORT_ORDER)
)
colors_c = [COHORT_COLORS[c] for c in COHORT_ORDER]
bars = axes[0].bar(range(len(COHORT_ORDER)), cohort_help['mean'],
                   color=colors_c, edgecolor='white', linewidth=1.2)
axes[0].set_xticks(range(len(COHORT_ORDER)))
axes[0].set_xticklabels(['얼리어답터\nD+0~30', '초기다수\nD+31~90', '후기다수\nD+91+'], fontsize=9)
axes[0].set_title('코호트별 평균 도움돼요', fontsize=12, fontweight='bold')
axes[0].set_ylabel('평균 helpfulTrueCount')
for bar, v in zip(bars, cohort_help['mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── days_since_release vs helpfulness 산점도 (전체) ──────────
ax2 = axes[1]
for prod in apple_prods[:2] + sam_prods[:2]:   # 대표 4개만
    sub = df_valid[(df_valid['product_label'] == prod) & (df_valid['helpfulTrueCount'] > 0)]
    if sub.empty: continue
    ax2.scatter(sub['days_t3'], sub['helpfulTrueCount'].clip(upper=50),
                alpha=0.3, s=10, color=BRAND_COLORS_T3[prod], label=prod)

# 전체 추세선 (binned mean)
trend = df_valid.groupby('days_bin')['helpfulTrueCount'].mean().reset_index()
ax2.plot(trend['days_bin'], trend['helpfulTrueCount'], 'k-', linewidth=2,
         label='전체 평균', zorder=10)
ax2.set_title('출시 후 경과 일수 vs 도움돼요\n(50 이상 clip)', fontsize=11, fontweight='bold')
ax2.set_xlabel('출시 후 경과 일수')
ax2.set_ylabel('helpfulTrueCount')
ax2.legend(fontsize=7, loc='upper right')

# ── 부정 리뷰의 코호트별 helpfulness (초기 부정이 더 공감받나?) ─
neg_help = (
    df_valid[df_valid['rating'] <= 2]
    [df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby('cohort')['helpfulTrueCount']
    .agg(['mean', 'count'])
    .reindex(COHORT_ORDER)
)
ax3 = axes[2]
bars3 = ax3.bar(range(len(COHORT_ORDER)), neg_help['mean'],
                color=colors_c, edgecolor='white', linewidth=1.2)
ax3.set_xticks(range(len(COHORT_ORDER)))
ax3.set_xticklabels(['얼리어답터\nD+0~30', '초기다수\nD+31~90', '후기다수\nD+91+'], fontsize=9)
ax3.set_title('부정 리뷰(1-2점)의 코호트별\n평균 도움돼요', fontsize=11, fontweight='bold')
ax3.set_ylabel('평균 helpfulTrueCount')
for bar, (mean_v, cnt) in zip(bars3, neg_help.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{mean_v:.1f}\n(n={int(cnt)})', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('output/t3_04_helpfulness.png', dpi=150, bbox_inches='tight')
plt.show()

# Kruskal-Wallis
samples = [df_valid[df_valid['cohort']==c]['helpfulTrueCount'].dropna().values
           for c in COHORT_ORDER if df_valid[df_valid['cohort']==c].shape[0] > 0]
stat, p = stats.kruskal(*samples)
print(f'Helpfulness 코호트 간 Kruskal-Wallis: H={stat:.3f}, p={p:.4f}')
print('→', '코호트 간 helpfulness 차이 유의' if p < 0.05 else '차이 없음')
print('\n코호트별 통계:')
print(cohort_help.to_string())

### 3-4. BERTopic `topics_over_time` — 토픽이 시간에 따라 어떻게 변하는가
> **LDA 대신 BERTopic Dynamic Topic Modeling 사용**  
> 각 토픽이 출시 후 경과 시간에 따라 얼마나 활발하게 논의되는지 추적

In [ ]:
import warnings, pickle; warnings.filterwarnings('ignore')
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer as CV_bert2, TfidfVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

# token_str이 df에 없으면 캐시에서 직접 복구
if 'token_str' not in df.columns:
    with open('output/morpheme_cache.pkl', 'rb') as f:
        _tok = pickle.load(f)
    df['tokens']    = _tok
    df['token_str'] = df['tokens'].apply(lambda t: ' '.join(t) if isinstance(t, list) else '')
    print('token_str 복구 완료')

# 출시 후 리뷰 + 토큰 있는 것만
_mask = (df['cohort'] != '출시 전') & (df['token_str'].fillna('').str.strip().str.len() > 0)
df_tot = df[_mask].copy().reset_index(drop=True)
if 'days_bin' not in df_tot.columns:
    df_tot['days_bin'] = (df_tot['days_t3'] // 10) * 10

tot_docs = df_tot['token_str'].tolist()
print(f'BERTopic 입력 문서: {len(tot_docs)}건')

tfidf_emb2 = TfidfVectorizer(token_pattern=r'[가-힣]{2,}',
                              max_features=2000, sublinear_tf=True)
emb2 = tfidf_emb2.fit_transform(tot_docs).toarray()

umap2    = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
hdbscan2 = HDBSCAN(min_cluster_size=15, min_samples=3,
                   metric='euclidean', cluster_selection_method='eom', prediction_data=True)
cv2      = CV_bert2(ngram_range=(1, 2), min_df=1, token_pattern=r'[가-힣]{2,}')
ctfidf2  = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model_t3 = BERTopic(
    language='multilingual',
    umap_model=umap2, hdbscan_model=hdbscan2,
    vectorizer_model=cv2, ctfidf_model=ctfidf2,
    nr_topics=8, verbose=False, calculate_probabilities=False,
)
topics_t3, _ = topic_model_t3.fit_transform(tot_docs, embeddings=emb2)
df_tot['bert_topic_t3'] = topics_t3

ti3 = topic_model_t3.get_topic_info()
print(f'\n토픽 수: {(ti3["Topic"]!=-1).sum()}개  |  Outlier: {(pd.Series(topics_t3)==-1).mean()*100:.1f}%')
print('\n토픽 요약:')
for _, row in ti3[ti3['Topic']!=-1].iterrows():
    top3 = ' / '.join([w for w, _ in topic_model_t3.get_topic(row['Topic'])[:3]])
    print(f'  T{row["Topic"]:2d} (n={row["Count"]:4d}): {top3}')

In [ ]:
# ── topics_over_time: 코호트 단위로 토픽 비중 변화 추적 ───────
# BERTopic topics_over_time은 datetime을 bins로 나눔
# days_t3를 직접 timestamps로 사용 (출시 기준 상대 시간)
# 단, topics_over_time은 실제 datetime을 요구하므로 days를 anchor date로 offset

ANCHOR = pd.Timestamp('2000-01-01')
df_tot['pseudo_dt'] = ANCHOR + pd.to_timedelta(df_tot['days_t3'].clip(lower=0), unit='D')

topics_over_time = topic_model_t3.topics_over_time(
    tot_docs,
    df_tot['pseudo_dt'].tolist(),
    nr_bins=15,          # 15개 시간 구간
    global_tuning=True,
    evolution_tuning=True,
)

# ── 시각화: 매뉴얼 (plotly 없이 matplotlib) ───────────────────
valid_topics_t3 = ti3[ti3['Topic'] != -1]['Topic'].tolist()
topic_labels_t3 = {}
for tid in valid_topics_t3:
    top3 = [w for w, _ in topic_model_t3.get_topic(tid)[:3] if isinstance(w, str)]
    topic_labels_t3[tid] = f'T{tid}: {"/".join(top3[:2])}'

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

cmap_t = plt.cm.tab10

for ax_idx, tid in enumerate(valid_topics_t3[:8]):
    ax = axes[ax_idx]
    topic_df = topics_over_time[topics_over_time['Topic'] == tid].copy()
    topic_df['days_approx'] = (topic_df['Timestamp'] - ANCHOR).dt.days

    ax.fill_between(topic_df['days_approx'], topic_df['Frequency'],
                    alpha=0.4, color=cmap_t(ax_idx))
    ax.plot(topic_df['days_approx'], topic_df['Frequency'],
            color=cmap_t(ax_idx), linewidth=2)

    for b, label in [(30, 'D+30'), (90, 'D+90')]:
        ax.axvline(b, color='gray', linestyle=':', linewidth=1)
        ax.text(b + 1, ax.get_ylim()[1] * 0.9, label, fontsize=7, color='gray')

    ax.set_title(topic_labels_t3.get(tid, f'T{tid}'), fontsize=9, fontweight='bold')
    ax.set_xlabel('출시 후 경과 일수')
    ax.set_ylabel('빈도')

for ax in axes[len(valid_topics_t3):]:
    ax.axis('off')

plt.suptitle('BERTopic `topics_over_time` — 토픽별 시간 변화\n(회색점선=코호트 경계)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_05_topics_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-5. KeyBERT 코호트별 핵심 키워드 + UMAP 임베딩 시각화
> **TF-IDF 워드클라우드 대신** BERT 임베딩 기반 의미 키워드 추출 (KeyBERT)  
> + 각 리뷰를 2D 공간에 투영해 코호트 간 의미적 분포 차이 시각화

In [ ]:
from keybert import KeyBERT

# ── KeyBERT 초기화 (다국어 모델) ─────────────────────────────
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

# 코호트별 대표 문서 생성 (토큰 문자열 합치기)
cohort_docs_kw = {}
for cohort in COHORT_ORDER:
    texts = df_tot[df_tot['cohort'] == cohort]['token_str'].tolist()
    # 랜덤 샘플링 후 합치기 (KeyBERT는 문서 단위로 동작)
    sample = np.random.default_rng(SEED).choice(texts, size=min(200, len(texts)), replace=False)
    cohort_docs_kw[cohort] = ' '.join(sample)

# ── KeyBERT 키워드 추출 ──────────────────────────────────────
print('KeyBERT 코호트별 핵심 키워드 (의미 기반):\n')
cohort_keywords_kb = {}
for cohort, doc in cohort_docs_kw.items():
    keywords = kw_model.extract_keywords(
        doc,
        keyphrase_ngram_range=(1, 2),
        stop_words=None,
        top_n=15,
        diversity=0.5,   # MMR 다양성 조절
        use_mmr=True,
    )
    cohort_keywords_kb[cohort] = keywords
    top_str = ', '.join([f'{kw}({sc:.3f})' for kw, sc in keywords[:10]])
    print(f'[{cohort}]\n  {top_str}\n')

# ── 코호트별 키워드 비교 시각화 ──────────────────────────────
fig, axes_kb = plt.subplots(1, 3, figsize=(18, 7))

for ax, cohort in zip(axes_kb, COHORT_ORDER):
    kws = cohort_keywords_kb[cohort][:12]
    words, scores = zip(*kws)
    color = COHORT_COLORS[cohort]

    bars = ax.barh(list(words)[::-1], list(scores)[::-1],
                   color=color, alpha=0.8, edgecolor='white', linewidth=1.2)
    ax.set_title(f'{cohort}\nKeyBERT 핵심 키워드', fontsize=10, fontweight='bold',
                 color=color)
    ax.set_xlabel('의미 유사도 점수')
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('코호트별 핵심 키워드 비교 (KeyBERT — BERT 임베딩 기반)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_06_keybert_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── UMAP 2D: 각 리뷰 임베딩 → 코호트별 분포 시각화 ──────────
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.1,
               metric='cosine', random_state=SEED)
emb_2d = umap_2d.fit_transform(emb2)
df_tot['umap_x'] = emb_2d[:, 0]
df_tot['umap_y'] = emb_2d[:, 1]

fig, axes_u = plt.subplots(1, 2, figsize=(16, 7))

# 코호트별 색상으로 산점도
ax_u1 = axes_u[0]
for cohort in COHORT_ORDER:
    sub = df_tot[df_tot['cohort'] == cohort]
    ax_u1.scatter(sub['umap_x'], sub['umap_y'],
                  c=COHORT_COLORS[cohort], label=cohort,
                  alpha=0.4, s=8, edgecolors='none')

# BERTopic 토픽별 중심 표시
for tid in valid_topics_t3:
    mask = df_tot['bert_topic_t3'] == tid
    if mask.sum() < 5: continue
    cx, cy = df_tot.loc[mask, 'umap_x'].mean(), df_tot.loc[mask, 'umap_y'].mean()
    lbl = topic_labels_t3.get(tid, f'T{tid}')
    ax_u1.annotate(lbl, (cx, cy), fontsize=7, ha='center',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax_u1.set_title('UMAP 2D — 코호트별 리뷰 의미 분포\n(BERTopic 토픽 중심 표시)',
                fontsize=11, fontweight='bold')
ax_u1.legend(markerscale=2, fontsize=9)
ax_u1.set_xlabel('UMAP 1')
ax_u1.set_ylabel('UMAP 2')

# 제품별 색상으로 산점도 (코호트 분포가 제품 특성에 따라 달라지나?)
ax_u2 = axes_u[1]
for prod in apple_prods + sam_prods:
    sub = df_tot[df_tot['product_label'] == prod]
    ax_u2.scatter(sub['umap_x'], sub['umap_y'],
                  c=BRAND_COLORS_T3[prod], label=prod,
                  alpha=0.3, s=8, edgecolors='none')

ax_u2.set_title('UMAP 2D — 제품별 리뷰 의미 분포', fontsize=11, fontweight='bold')
ax_u2.legend(markerscale=2, fontsize=8, ncol=2)
ax_u2.set_xlabel('UMAP 1')
ax_u2.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig('output/t3_07_umap_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-6. 종합 인사이트 — 리뷰 라이프사이클 비즈니스 가치

In [ ]:
print('=' * 70)
print('Topic 3 — 리뷰 라이프사이클 종합 인사이트')
print('=' * 70)

# 전체 코호트 분포
n_early = (df_valid['cohort'] == COHORT_ORDER[0]).sum()
n_mid   = (df_valid['cohort'] == COHORT_ORDER[1]).sum()
n_late  = (df_valid['cohort'] == COHORT_ORDER[2]).sum()
n_total = n_early + n_mid + n_late
print(f'\n코호트 분포: 얼리어답터 {n_early}건({n_early/n_total*100:.1f}%) | '
      f'초기다수 {n_mid}건({n_mid/n_total*100:.1f}%) | '
      f'후기다수 {n_late}건({n_late/n_total*100:.1f}%)')

# 코호트별 평균 평점
for cohort in COHORT_ORDER:
    sub = df_valid[df_valid['cohort'] == cohort]
    print(f'  {cohort}: 평균 평점 {sub["rating"].mean():.2f} | '
          f'부정비율 {(sub["rating"]<=2).mean()*100:.1f}% | '
          f'평균도움돼요 {sub["helpfulTrueCount"].mean():.2f}')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 1. 허니문 효과는 제품마다 다르다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Galaxy Z Flip7: 얼리어답터 평점이 후기다수보다 낮음 → 역허니문
  → 출시 초기 열망 구매 후 실망 → 포폼팩터 적응 문제
• iPhone 17 Pro: 얼리어답터가 더 냉정 → 열렬한 팬보다 실용적 얼리어답터
• 허니문이 강한 제품 = 팬덤 기반 마케팅이 효과적
• 역허니문 제품 = 온보딩 개선이 초기 만족도 방어에 중요

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 2. 배송 불만은 출시 직후에 집중된다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• BERTopic topics_over_time: T1(박스/포장 불만)이 D+0~30에 집중
• 출시 초기 물량 급증 → 쿠팡 물류 과부하 → 포장 부실
• 비즈니스 제안: 신제품 출시 첫 달 물류 전담팀 배치 / 프리미엄 포장 옵션 도입

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 3. 초기 리뷰가 플랫폼 평점에 미치는 영향은 제한적
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 허니문 효과 통계 유의 제품 없음 (Mann-Whitney p > 0.05 전 제품)
• 초기 리뷰가 구조적으로 평점을 왜곡하지는 않음
• 단, helpfulness는 초기 리뷰가 높을 수 있음 → '유용한 리뷰' 알고리즘 영향 가능

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 4. KeyBERT로 드러난 코호트별 관심사 이동
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 얼리어답터 → "갈아탔다", "비교", "기다렸던" (전환·기대 중심)
• 초기다수  → "화면", "카메라", "성능" (기능 실용성 중심)
• 후기다수  → "배터리", "발열", "오래쓰니" (장기 사용성 중심)
→ 출시 타이밍별로 다른 마케팅 메시지가 필요함
""")
print('분석 방법: BERTopic topics_over_time + KeyBERT (paraphrase-multilingual-MiniLM) + UMAP 2D')

---
### 2-9. 형태소 분석 기반 TF-IDF (Kiwi) — 워드클라우드 고도화

> 공백 분리의 한계를 극복: 명사(NNG/NNP) + 형용사(VA) 단위로 정밀 추출

In [ ]:
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords

kiwi = Kiwi()

# 분석에 쓸 품사 (명사류 + 형용사)
KEEP_POS = {'NNG', 'NNP', 'SL', 'VA'}  # 일반명사, 고유명사, 외래어, 형용사

# 도메인·제품명 등 분석에 무의미한 토큰
CUSTOM_STOP = {
    '아이폰', '갤럭시', '폰', '핸드폰', '스마트폰', '제품', '구매', '배송',
    '리뷰', '후기', '구입', '사용', '이것', '저것', '그것', '하나',
    '생각', '정말', '진짜', '너무', '이번', '것', '수', '때', '제',
    '분', '번', '개', '명', '원', '년', '월', '일',
}

def extract_morphemes(text: str) -> list[str]:
    """Kiwi로 형태소 분석 → 명사/형용사 추출"""
    if not isinstance(text, str) or len(text.strip()) < 2:
        return []
    tokens = []
    for sent in kiwi.analyze(text[:2000]):  # 너무 긴 텍스트는 2000자로 제한
        for tok in sent[0]:
            form = tok.form.strip()
            tag = str(tok.tag)
            if tag in KEEP_POS and len(form) >= 2 and form not in CUSTOM_STOP:
                tokens.append(form)
        break  # 첫 번째 분석 결과만 사용
    return tokens

# ── 전체 리뷰 형태소 분석 (캐싱) ────────────────────────────
import pickle, os

CACHE_PATH = 'output/morpheme_cache.pkl'
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        df_tokens = pickle.load(f)
    print(f'캐시 로드 완료: {len(df_tokens)}건')
else:
    print('형태소 분석 시작... (1~2분 소요)')
    from tqdm import tqdm
    tqdm.pandas()
    df_tokens = df['content'].fillna('').progress_apply(extract_morphemes)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(df_tokens, f)
    print('분석 완료 & 캐시 저장')

df['tokens'] = df_tokens
df['token_str'] = df['tokens'].apply(lambda t: ' '.join(t))

# 샘플 확인
print('\n샘플 형태소 분석:')
for _, row in df[df['content_len'] > 100].sample(2, random_state=SEED).iterrows():
    print(f'  원문: {str(row["content"])[:80]}...')
    print(f'  토큰: {row["tokens"][:15]}\n')

In [ ]:
# ── 형태소 기반 TF-IDF 재계산 ────────────────────────────────
vectorizer_kiwi = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=5000,
    min_df=2,
    sublinear_tf=True,
)

group_corpus_kiwi = {
    g: ' '.join(df[df['domain_group'] == g]['token_str'].tolist())
    for g in main_groups_tfidf
}
tfidf_kiwi = vectorizer_kiwi.fit_transform(list(group_corpus_kiwi.values()))
feat_kiwi = vectorizer_kiwi.get_feature_names_out()

TOP_N = 25
group_kw_kiwi = {}
for i, g in enumerate(main_groups_tfidf):
    scores = tfidf_kiwi[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:TOP_N]
    group_kw_kiwi[g] = [(feat_kiwi[j], float(scores[j])) for j in top_idx]

# ── 워드클라우드 (형태소 버전) ────────────────────────────────
fig, axes_wc2 = plt.subplots(2, 2, figsize=(16, 12))
axes_wc2 = axes_wc2.flatten()

CMAPS = {'네이버': 'Greens', 'Gmail': 'Reds', '카카오': 'YlOrBr', '네이트': 'Oranges'}

for ax, g in zip(axes_wc2, main_groups_tfidf):
    kw_dict = {kw: sc for kw, sc in group_kw_kiwi[g]}
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color='white',
        width=700, height=450,
        max_words=50,
        colormap=CMAPS.get(g, 'Set2'),
        random_state=SEED,
        prefer_horizontal=0.85,
    ).generate_from_frequencies(kw_dict)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'[{g}] 핵심 키워드 (형태소 분석)', fontsize=13, fontweight='bold',
                 color=GROUP_COLORS[g], pad=8)

plt.suptitle('도메인 그룹별 핵심 키워드 — Kiwi 형태소 분석 기반 TF-IDF', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t2_08_wordcloud_kiwi.png', dpi=150, bbox_inches='tight')
plt.show()

# 상위 10개 텍스트 출력
for g in main_groups_tfidf:
    top10 = [f'{kw}({sc:.3f})' for kw, sc in group_kw_kiwi[g][:10]]
    print(f'[{g}]: {", ".join(top10)}')

### 2-10. LDA 토픽 모델링 — 도메인 그룹별 잠재 주제 비교

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

N_TOPICS = 5
N_TOP_WORDS = 8

# 각 그룹별 LDA 독립 실행
lda_results = {}  # g -> (lda, vectorizer, topic_words)

fig, axes_lda = plt.subplots(len(main_groups_tfidf), N_TOPICS,
                              figsize=(18, 3.2 * len(main_groups_tfidf)))

for row_i, g in enumerate(main_groups_tfidf):
    docs = df[df['domain_group'] == g]['token_str'].tolist()
    docs = [d for d in docs if d.strip()]

    cv = CountVectorizer(max_features=2000, min_df=2, ngram_range=(1, 1))
    dtm = cv.fit_transform(docs)
    vocab = cv.get_feature_names_out()

    lda = LatentDirichletAllocation(
        n_components=N_TOPICS,
        random_state=SEED,
        max_iter=30,
        learning_method='online',
    )
    lda.fit(dtm)

    topic_words = []
    for t_idx, topic in enumerate(lda.components_):
        top_w = [vocab[i] for i in topic.argsort()[:-N_TOP_WORDS-1:-1]]
        topic_words.append(top_w)

    lda_results[g] = (lda, cv, topic_words)

    # 시각화: 토픽별 상위 단어 막대
    for t_idx, (top_w, ax) in enumerate(zip(topic_words, axes_lda[row_i])):
        scores = lda.components_[t_idx]
        word_scores = [(w, scores[list(vocab).index(w)]) for w in top_w]
        words, vals = zip(*word_scores)
        ax.barh(list(words)[::-1], list(vals)[::-1],
                color=GROUP_COLORS[g], alpha=0.8, edgecolor='white')
        ax.set_title(f'[{g}] Topic {t_idx+1}', fontsize=9, fontweight='bold',
                     color=GROUP_COLORS[g])
        ax.tick_params(axis='y', labelsize=8)
        ax.tick_params(axis='x', labelsize=7)
        ax.set_xlabel('weight', fontsize=7)

plt.suptitle('LDA 토픽 모델링 — 도메인 그룹별 (형태소 분석 기반)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t2_09_lda.png', dpi=150, bbox_inches='tight')
plt.show()

# 텍스트 요약
print('=' * 60)
for g in main_groups_tfidf:
    _, _, tw = lda_results[g]
    print(f'\n[{g}] LDA 토픽 요약:')
    for i, words in enumerate(tw):
        print(f'  Topic {i+1}: {", ".join(words)}')

### 2-11. BERTopic — 전체 리뷰 의미 기반 토픽 + 도메인 그룹별 분포

In [ ]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer as CV_bert, TfidfVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import warnings
warnings.filterwarnings('ignore')

bertopic_docs = df['token_str'].fillna('').tolist()

# ── TF-IDF 임베딩 (한국어 적합) ──────────────────────────────
tfidf_emb = TfidfVectorizer(
    token_pattern=r'[가-힣]{2,}',
    max_features=2000, sublinear_tf=True,
)
embeddings = tfidf_emb.fit_transform(bertopic_docs).toarray()
print(f'임베딩 shape: {embeddings.shape}')

umap_model = UMAP(
    n_neighbors=10, n_components=5,
    min_dist=0.0, metric='cosine', random_state=SEED,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=15, min_samples=3,
    metric='euclidean', cluster_selection_method='eom',
    prediction_data=True,
)
cv_bert = CV_bert(
    ngram_range=(1, 2), min_df=1,
    token_pattern=r'[가-힣]{2,}',
)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    language='multilingual',        # 한글 유지 — 'english'는 한글을 제거함
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=cv_bert,
    ctfidf_model=ctfidf,
    nr_topics=8,                    # 53개 초기 토픽 → 8개로 병합
    verbose=False,
    calculate_probabilities=False,
)

topics, _ = topic_model.fit_transform(bertopic_docs, embeddings=embeddings)
df['bert_topic'] = topics

topic_info = topic_model.get_topic_info()
n_real = (topic_info['Topic'] != -1).sum()
print(f'발견된 토픽 수: {n_real}')
print(f'Outlier(-1) 비율: {(pd.Series(topics) == -1).mean()*100:.1f}%')
print('\n상위 토픽:')
print(topic_info[['Topic','Count','Name']].to_string())

In [ ]:
# ── BERTopic 시각화 1: 토픽별 상위 단어 (barchart) ──────────
topic_info = topic_model.get_topic_info()
valid_topics = topic_info[topic_info['Topic'] != -1]['Topic'].tolist()
n_show = min(8, len(valid_topics))
show_topics = valid_topics[:n_show]

fig, axes_bt = plt.subplots(2, 4, figsize=(18, 9))
axes_bt = axes_bt.flatten()

for ax, tid in zip(axes_bt, show_topics):
    words_scores = [(w, v) for w, v in topic_model.get_topic(tid) if v > 0 and isinstance(w, str) and w.strip()]
    if not words_scores:
        ax.set_title(f'Topic {tid} (빈 토픽)', fontsize=9)
        ax.axis('off')
        continue
    words_scores = words_scores[:8]
    words, vals = zip(*words_scores)
    ax.barh(list(words)[::-1], list(vals)[::-1], color='steelblue', alpha=0.8, edgecolor='white')
    cnt = topic_info[topic_info['Topic'] == tid]['Count'].values[0]
    ax.set_title(f'Topic {tid}  (n={cnt})', fontsize=10, fontweight='bold')
    ax.tick_params(axis='y', labelsize=9)
    ax.tick_params(axis='x', labelsize=8)

for ax in axes_bt[n_show:]:
    ax.axis('off')

plt.suptitle('BERTopic 상위 토픽 키워드 (한글 형태소 기반)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t2_10_bertopic_words.png', dpi=150, bbox_inches='tight')
plt.show()

# 토픽 이름 붙이기 (상위 3단어)
topic_labels = {
    tid: f'T{tid}: ' + '/'.join([w for w, _ in topic_model.get_topic(tid)[:3] if isinstance(w,str)])
    for tid in valid_topics
}
print('토픽 레이블:')
for tid, lbl in list(topic_labels.items())[:10]:
    cnt = topic_info[topic_info['Topic']==tid]['Count'].values[0]
    print(f'  {lbl}  (n={cnt})')

In [ ]:
# ── BERTopic 시각화 2: 도메인 그룹별 토픽 분포 히트맵 ─────────
df_bt = df[df['bert_topic'] != -1].copy()

# 토픽에 레이블 붙이기 (상위 3단어로)
topic_labels = {}
for tid in valid_topics:
    top3 = [w for w, _ in topic_model.get_topic(tid)[:3]]
    topic_labels[tid] = f'T{tid}: {"/".join(top3)}'

df_bt['topic_label'] = df_bt['bert_topic'].map(
    lambda t: topic_labels.get(t, f'T{t}')
)

# 도메인 그룹 × 토픽 cross-tab
used_topics = [t for t in valid_topics if t in df_bt['bert_topic'].values]
topic_cross = (
    df_bt[df_bt['bert_topic'].isin(used_topics)]
    .groupby(['domain_group', 'topic_label'])
    .size().unstack(fill_value=0)
    .reindex([g for g in main_groups_tfidf if g in df_bt['domain_group'].values])
)
topic_cross_pct = topic_cross.div(topic_cross.sum(axis=1), axis=0) * 100

fig, axes_cross = plt.subplots(1, 2, figsize=(18, 5))

# 히트맵
sns.heatmap(topic_cross_pct, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=axes_cross[0], cbar_kws={'label': '%'})
axes_cross[0].set_title('도메인 그룹별 BERTopic 분포 (%)', fontsize=12, fontweight='bold')
axes_cross[0].set_xlabel('')
axes_cross[0].set_ylabel('')
axes_cross[0].tick_params(axis='x', rotation=30)

# 스택 막대 (각 그룹에서 토픽 비중)
topic_cross_pct.T.plot(kind='bar', ax=axes_cross[1],
                        color=[GROUP_COLORS[g] for g in topic_cross_pct.index],
                        edgecolor='white', width=0.7)
axes_cross[1].set_title('토픽별 도메인 그룹 기여 비율', fontsize=12, fontweight='bold')
axes_cross[1].set_xlabel('')
axes_cross[1].set_ylabel('%')
axes_cross[1].tick_params(axis='x', rotation=30)
axes_cross[1].legend(title='그룹', fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('output/t2_11_bertopic_group.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n[그룹별 가장 지배적인 BERTopic]')
for g in main_groups_tfidf:
    if g not in topic_cross_pct.index:
        continue
    top_topic = topic_cross_pct.loc[g].idxmax()
    top_pct = topic_cross_pct.loc[g].max()
    print(f'  {g}: {top_topic}  ({top_pct:.1f}%)')

### 2-12. 불만 카테고리 분류 — 도메인 그룹별 불만 구조 비교

In [ ]:
# ── 불만 카테고리 키워드 사전 ────────────────────────────────
COMPLAINT_CATS = {
    '📦 배송/포장':   ['배송', '포장', '박스', '택배', '배달', '찌그러', '뜯기', '흠집', '스크래치', '완충'],
    '💰 가격/가성비': ['비싸', '가격', '가성비', '환불', '반품', '돈이', '가치'],
    '🔋 배터리/발열': ['배터리', '발열', '뜨거', '충전', '방전', '소모', '오버히트', '과열'],
    '📷 카메라':      ['카메라', '사진', '화질', '렌즈', '흔들림', '노이즈', '야간', '화소'],
    '⚡ 성능/속도':   ['버벅', '느린', '속도', '끊김', '지연', '프리징', '멈춤', '렉'],
    '🔧 A/S/품질':   ['고장', 'AS', '서비스', '수리', '센터', '결함', '불량', '흠'],
    '💾 소프트웨어':  ['업데이트', '앱', '오류', '버그', '소프트웨어', '인터페이스', '설정', 'UI'],
}

def classify_complaints(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    cats = []
    for cat, kws in COMPLAINT_CATS.items():
        if any(kw in text for kw in kws):
            cats.append(cat)
    return cats if cats else ['기타']

# 부정 리뷰(1-2점)만 분류
neg_df = df[df['rating'] <= 2].copy()
neg_df['complaint_cats'] = neg_df['content'].apply(classify_complaints)

# 그룹 × 카테고리 매트릭스
rows = []
for _, row in neg_df.iterrows():
    for cat in row['complaint_cats']:
        rows.append({'domain_group': row['domain_group'], 'category': cat})
comp_df = pd.DataFrame(rows)

comp_cross = (
    comp_df[comp_df['domain_group'].isin(main_groups_tfidf)]
    .groupby(['domain_group', 'category'])
    .size().unstack(fill_value=0)
    .reindex(main_groups_tfidf)
    .fillna(0)
)
# 비율 (해당 그룹 부정 리뷰 수 대비)
neg_counts = {g: len(neg_df[neg_df['domain_group'] == g]) for g in main_groups_tfidf}
comp_pct = comp_cross.copy().astype(float)
for g in main_groups_tfidf:
    if neg_counts[g] > 0:
        comp_pct.loc[g] = comp_cross.loc[g] / neg_counts[g] * 100

print('부정 리뷰 수:', neg_counts)
print('\n불만 카테고리 비율 (%):')
print(comp_pct.round(1).to_string())

# ── 시각화 ────────────────────────────────────────────────────
fig, axes_comp = plt.subplots(1, 2, figsize=(18, 6))

# 히트맵
sns.heatmap(comp_pct.T, annot=True, fmt='.1f', cmap='Reds',
            linewidths=0.5, ax=axes_comp[0], cbar_kws={'label': '비율 (%)'})
axes_comp[0].set_title('도메인 그룹별 불만 카테고리 비율\n(부정 리뷰 대비 %)', fontsize=12, fontweight='bold')
axes_comp[0].set_ylabel('')
axes_comp[0].set_xlabel('')

# 그룹별 상위 불만 비교 (grouped bar)
comp_pct.plot(kind='bar', ax=axes_comp[1], color=plt.cm.Set3.colors[:len(comp_pct.columns)],
              edgecolor='white', width=0.75)
axes_comp[1].set_title('그룹별 불만 카테고리 분포', fontsize=12, fontweight='bold')
axes_comp[1].set_xlabel('')
axes_comp[1].set_ylabel('부정 리뷰 대비 %')
axes_comp[1].tick_params(axis='x', rotation=0)
axes_comp[1].legend(loc='upper right', fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('output/t2_12_complaint.png', dpi=150, bbox_inches='tight')
plt.show()

# 각 그룹의 1등 불만
print('\n[그룹별 가장 많은 불만 카테고리]')
for g in main_groups_tfidf:
    if g in comp_pct.index and comp_pct.loc[g].sum() > 0:
        top_cat = comp_pct.loc[g].idxmax()
        top_pct = comp_pct.loc[g].max()
        print(f'  {g} (부정 리뷰 {neg_counts[g]}건): {top_cat} ({top_pct:.1f}%)')